In [3]:
import sys
import numpy as np
from pathlib import Path

project_root = Path("..").resolve()
sys.path.append(str(project_root))

from embedder import Embedder
from gitsource import GithubRepositoryDataReader

model_path = project_root / "models" / "Xenova" / "all-MiniLM-L6-v2"
embedder = Embedder(path=str(model_path))

In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

### Q1

In [5]:
vector = embedder.encode("How does approximate nearest neighbor search work?")
print(vector[0])

-0.02058203437252893


### Q2

In [6]:
query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

page_reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: path.endswith("02-vector-search/lessons/07-sqlitesearch-vector.md"),
)

page = [file.parse() for file in page_reader.read()][0]
content = page["content"]

# tokenizer.json caps at 128 tokens by default; increase to cover the ANN section
embedder.tokenizer.enable_truncation(max_length=256)
content_vector = embedder.encode(content)
embedder.tokenizer.enable_truncation(max_length=128)  # restore default

similarity = float(np.dot(query_vector, content_vector))
print("Cosine similarity:", similarity)

Cosine similarity: 0.4568133844975112


In [ ]:
from gitsource import chunk_documents

# size=2000  → each chunk is at most 2 000 characters
# step=1000  → the next chunk starts 1 000 chars later (50 % overlap)
# Each chunk dict keeps the parent document's metadata (e.g. filename)
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"{len(chunks)} chunks from {len(documents)} documents")

### Q3 — Chunking and search by hand

In [8]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch_texts = [chunk["content"] for chunk in chunks[i : i + batch_size]]
    batch_vectors = embedder.encode_batch(batch_texts)  # shape: (batch, 384)
    vectors.extend(batch_vectors)                       # append each row

# Stack into a 2-D matrix: rows = chunks, columns = vector dimensions
X = np.array(vectors)
print(f"Matrix shape: {X.shape}  (chunks × embedding dimensions)")

  0%|          | 0/6 [00:00<?, ?it/s]

Matrix shape: (295, 384)  (chunks × embedding dimensions)


In [9]:
# v is the Q1 query vector (already computed above as `vector`)
# X.dot(v) multiplies every row of X by v and sums — one cosine similarity per chunk
v = vector
scores = X.dot(v)

best_idx = scores.argmax()
print(f"Best score : {scores[best_idx]:.4f}")
print(f"Filename   : {chunks[best_idx]['filename']}")

Best score : 0.6489
Filename   : 02-vector-search/lessons/07-sqlitesearch-vector.md


### Q4 — Vector search with minsearch

In [10]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)
print("Index ready")

Index ready


In [11]:
q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)

results = vindex.search(q4_vector, num_results=5)

print("Top results:")
for i, r in enumerate(results):
    print(f"  {i+1}. {r['filename']}")

print(f"\nAnswer (first result filename): {results[0]['filename']}")

Top results:
  1. 04-evaluation/lessons/05-search-metrics.md
  2. 04-evaluation/lessons/01-intro.md
  3. 01-agentic-rag/lessons/05-search.md
  4. 04-evaluation/lessons/01-intro.md
  5. 04-evaluation/lessons/15-next-steps.md

Answer (first result filename): 04-evaluation/lessons/05-search-metrics.md


### Q5 — Text search vs vector search

**The core difference:**
- **Text search (`Index`)** — tokenizes the query and the documents, counts word matches (TF-IDF style). It only finds chunks that contain the *exact words* from the query.
- **Vector search (`VectorSearch`)** — converts query and documents to embeddings and measures *semantic similarity*. It can find chunks that talk about the same concept even if they use different words.

**The plan:**
1. Build a text index with `Index` over the same `chunks` we have from Q3 (using `content` as the text field).
2. Run both searches for *"How do I store vectors in PostgreSQL?"*.
3. Collect the top-5 filenames from each method.
4. The answer is the filename that appears in the vector results **but not** in the text results — the chunk the keyword search missed because it expressed the idea differently.

In [12]:
from minsearch import Index

# Index takes a list of dicts directly — no matrix needed.
# text_fields=["content"] tells it which field to tokenize and search.
# keyword_fields=["filename"] lets us see which file each result came from.
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)
print("Text index ready")

Text index ready


In [13]:
q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(q5_query)

# Vector search — uses semantic similarity (dot product on embeddings)
vec_results = vindex.search(q5_vector, num_results=5)
vec_files = [r["filename"] for r in vec_results]

# Text search — uses keyword matching (TF-IDF); takes the raw query string
txt_results = tindex.search(q5_query, num_results=5)
txt_files = [r["filename"] for r in txt_results]

print("Vector search top 5:")
for f in vec_files:
    print(f"  {f}")

print("\nText search top 5:")
for f in txt_files:
    print(f"  {f}")

# The answer: files returned by vector search but missed by text search
only_in_vector = set(vec_files) - set(txt_files)
print(f"\nIn vector results but NOT in text results: {only_in_vector}")

Vector search top 5:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md

Text search top 5:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md

In vector results but NOT in text results: {'02-vector-search/lessons/08-pgvector.md'}


### Q6 — Hybrid search with RRF

In [14]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [15]:
q6_query = "How do I give the model access to tools?"
q6_vector = embedder.encode(q6_query)

vector_results = vindex.search(q6_vector, num_results=5)
text_results = tindex.search(q6_query, num_results=5)

print("Vector results:", [r["filename"] for r in vector_results])
print("Text results:  ", [r["filename"] for r in text_results])

fused = rrf([vector_results, text_results])
print("\nAfter RRF:")
for i, r in enumerate(fused):
    print(f"  {i+1}. {r['filename']}")

Vector results: ['01-agentic-rag/lessons/01-intro.md', '04-evaluation/lessons/02-ground-truth.md', '01-agentic-rag/lessons/16-other-frameworks.md', '01-agentic-rag/lessons/15-frameworks.md', '01-agentic-rag/lessons/13-function-calling.md']
Text results:   ['01-agentic-rag/lessons/14-agentic-loop.md', '01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/13-function-calling.md', '04-evaluation/lessons/02-ground-truth.md']

After RRF:
  1. 01-agentic-rag/lessons/13-function-calling.md
  2. 01-agentic-rag/lessons/01-intro.md
  3. 01-agentic-rag/lessons/14-agentic-loop.md
  4. 04-evaluation/lessons/02-ground-truth.md
  5. 01-agentic-rag/lessons/16-other-frameworks.md
